# LangGraph without calling OpenAI

LG uses 5 steps to setup and run the framework
1. Define the state Object
2. Start Graph Builder
3. Create a node (function)
4. Create edges (decides which node to call)
5. Compile the Graph


In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel


In [ ]:
# 1. define state object using pydantic
class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
# 2. create graph_builder
graph_builder = StateGraph(State)

In [ ]:
# 3. Create node.  Call LLM here
llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

In [ ]:
# 4. add edgens.  START and END are defined in LG library
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [ ]:
# 5. Compile graph and display it as an img
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

## Put it all together in Gradio UI

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content


gr.ChatInterface(chat).launch()